In [14]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb xgboost

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


walmart-recruiting-store-sales-forecasting.zip: Skipping, found more recently modified local copy (use --force to force download)
2026/07/11 05:41:08 INFO mlflow.store.db.utils: Updating database tables


In [15]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

FIXED = dict(n_jobs=-1, tree_method='hist')

In [16]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

((421570, 5), (8190, 12), (45, 3))

In [17]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [18]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
]

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        return df[FEATURE_COLS]

In [19]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('XGBoost_v2_Training')

<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/7', creation_time=1783747303383, effective_trace_archival_retention=None, experiment_id='7', last_update_time=1783747303383, lifecycle_stage='active', name='XGBoost_v2_Training', tags={}, trace_location=None, workspace='default'>

In [20]:
with mlflow.start_run(run_name='XGBoost_v2_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_v2_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

missing_markdown_pct,▁
negative_sales_rows,▁
missing_markdown_pct,0.55849
negative_sales_rows,1285


In [21]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [10]:
SEARCH_SPACE = {
    'n_estimators': [300, 500, 800],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_weight': [1, 5, 10],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}
N_TRIALS = 12

rng = np.random.default_rng(42)

def sample_params(rng):
    return {k: v[rng.integers(len(v))] for k, v in SEARCH_SPACE.items()}

builder = FeatureBuilder(stores, features)
fold_data = []
for train_end, val_end in splits:
    tm = train['Date'] <= train_end
    vm = (train['Date'] > train_end) & (train['Date'] <= val_end)
    fold_data.append((
        builder.transform(train[tm]), y_train[tm],
        builder.transform(train[vm]), y_train[vm],
        train.loc[vm, 'IsHoliday'].values,
    ))

def cv_score(params):
    scores = []
    for X_tr, y_tr, X_val, y_val, hol in fold_data:
        model = XGBRegressor(**params, **FIXED)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(wmae(y_val.values, preds, hol))
    return scores

with mlflow.start_run(run_name='XGBoost_v2_Tuning'):
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_v2_Tuning', reinit='finish_previous')

    best_wmae, best_params = None, None
    for t in range(N_TRIALS):
        params = sample_params(rng)
        scores = cv_score(params)
        mean = float(np.mean(scores))
        mlflow.log_metric('trial_wmae_mean', mean, step=t)
        wandb.log({'trial': t, 'trial_wmae_mean': mean, **params})
        if best_wmae is None or mean < best_wmae:
            best_wmae, best_params = mean, params
        print(t, round(mean, 1), params)

    mlflow.log_metric('best_wmae_mean', best_wmae)
    mlflow.log_dict(best_params, 'best_params.json')
    wandb.log({'best_wmae_mean': best_wmae})
    wandb.finish()

best_wmae, best_params

0 2971.5 {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.05, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 1.0}
1 3078.7 {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.03, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 1.0}
2 3003.4 {'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.1, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.7}
3 3109.1 {'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 1.0}
4 3082.2 {'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.05, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8}
5 4521.4 {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 5, 'subsample': 1.0, 'colsample_bytree': 0.7}
6 2883.6 {'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.03, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 1.0}
7 3226.5 {'n_estimators': 800, 'max_depth':

best_wmae_mean,▁
colsample_bytree,██▁█▃▁██▃█▁▁
learning_rate,▃▁█▃▃▁▁▁█▃▃▁
max_depth,███▅▅▁█▅█▁█▅
min_child_weight,▄▁█▄█▄▄█▁▁█▄
n_estimators,▁▁███▄███▄██
subsample,▃▃▃▁▃█▁▃▃█▃█
trial,▁▂▂▃▄▄▅▅▆▇▇█
trial_wmae_mean,▁▂▂▂▂█▁▂▁▅▁▃
best_wmae_mean,2883.55848
colsample_bytree,0.7


(2883.558476219258,
 {'n_estimators': 800,
  'max_depth': 10,
  'learning_rate': 0.03,
  'min_child_weight': 5,
  'subsample': 0.7,
  'colsample_bytree': 1.0})

In [11]:
with mlflow.start_run(run_name='XGBoost_v2_CV'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_v2_CV', config=best_params, reinit='finish_previous')

    fold_scores = cv_score(best_params)
    for i, s in enumerate(fold_scores):
        mlflow.log_metric('wmae', s, step=i)
        wandb.log({'fold': i, 'wmae': s})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

fold,▁▅█
wmae,█▃▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,2162.36563
wmae_mean,2883.55848
wmae_std,751.66604


[np.float64(3920.4783485363023),
 np.float64(2567.8314454957876),
 np.float64(2162.365634625684)]

In [12]:
with mlflow.start_run(run_name='XGBoost_v2_Final'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_v2_Final', config=best_params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilder(stores, features)),
        ('model', XGBRegressor(**best_params, **FIXED)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/11 05:40:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_wmae,▁
train_wmae,1366.68828


np.float64(1366.688278551473)

In [13]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_XGBoost_v2.ipynb in Colab"
    !git push

[main c73df74] Run model_experiment_XGBoost_v2.ipynb in Colab
 7 files changed, 61 insertions(+)
 create mode 100644 mlruns/7/8266873a017648a0aba69c5e79b11398/artifacts/best_params.json
 create mode 100644 mlruns/7/models/m-b5e9b3d0743047599cc146d79eb48345/artifacts/MLmodel
 create mode 100644 mlruns/7/models/m-b5e9b3d0743047599cc146d79eb48345/artifacts/conda.yaml
 create mode 100644 mlruns/7/models/m-b5e9b3d0743047599cc146d79eb48345/artifacts/model.pkl
 create mode 100644 mlruns/7/models/m-b5e9b3d0743047599cc146d79eb48345/artifacts/python_env.yaml
 create mode 100644 mlruns/7/models/m-b5e9b3d0743047599cc146d79eb48345/artifacts/requirements.txt
Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (16/16), 13.75 MiB | 4.13 MiB/s, done.
Total 16 (delta 3), reused 2 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://g